# Exploring a large Abaqus mesh with pyramids-mesh

**Abaqus** (Dassault Systèmes) `.inp` input decks are one of the most widely used finite-element
formats: a `*NODE` block lists coordinates and `*ELEMENT, TYPE=...` blocks list connectivity. The
`abaqus` driver parses that keyword grammar into a pyramids `MeshDataset`.

This notebook is different from the small-patch tour: our file is a **real, large** engineering mesh —
about **three-quarters of a million nodes** and **a third of a million faces** (~94 MB on disk). At that
size you cannot simply draw the whole thing; the lesson here is **how to work with scale**: read the
file, inspect its size and properties, then **`crop` to a small window** and explore the subset with the
full pyramids toolbox.

## 1. Imports

`MeshDataset` is the object every driver hands back. Importing from `pyramids` also puts its vendored
GDAL/PROJ data in place, so nothing here is order-sensitive in a notebook — just import the names you
use. `add_tiles` would add a web basemap *when a mesh is georeferenced*; an engineering `.inp` never is,
so we only ever call it behind an `if ds.epsg` guard.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

from pyramids.netcdf.ugrid import MeshDataset
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the example file

This notebook lives in `docs/examples/real-data/`, so the shared `examples/data/real-data/` corpus is
three levels up. Paths inside a notebook are written **relative to the notebook itself**, and pyramids'
`read_file` accepts a `pathlib.Path` directly — we pass the `Path`, never `str(path)`.

In [ ]:
DATA = Path("../../../examples/data/real-data")
inp_file = DATA / "abaqus" / "1.inp"
inp_file, inp_file.exists(), f"{inp_file.stat().st_size / 1e6:.1f} MB"

## 3. Read the mesh — and time it

A 94 MB keyword deck sounds slow to parse, but the driver reads it in **a few seconds**. We wrap the read
in a timer so the cost is explicit. `MeshDataset.read_file(path, driver="abaqus")` returns a
`MeshDataset` — the same pyramids type a native UGRID NetCDF file produces.

In [ ]:
t0 = time.perf_counter()
ds = MeshDataset.read_file(inp_file, driver="abaqus")
read_seconds = time.perf_counter() - t0
print(f"read in {read_seconds:.1f} s  ->  {type(ds).__name__}")

Parsing the whole deck took only a handful of seconds. The one-time warning about reversed face
windings is expected: the driver normalises every face to counter-clockwise order on read, so a writer
emits a consistent orientation later.

## 4. Choose the plotting backend

cleopatra switches matplotlib to a headless `Agg` backend on the first `read_file`. Immediately after
that first read we call cleopatra's own chooser once: it detects the notebook and selects the inline
backend (and would fall back to `Agg` in a plain script or in CI), so every figure below embeds
natively.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 5. How big is it?

The scalar properties describe the whole mesh at a glance: `n_node` / `n_face` (size), the **connectivity
width** `k` (the widest face's node count), `crs` / `epsg` (georeferencing, if any), `bounds` (extent), and
`data_variable_names` (attached fields).

In [ ]:
conn = ds.mesh.face_node_connectivity.data
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "connectivity_width_k": conn.shape[1],
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(b, 1) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

This is a **large** mesh: **752 666 nodes** and **332 200 faces**. The connectivity width `k = 8`
means the face table has eight columns — the deck mixes element types, and a face with fewer nodes pads
its spare slots. `crs`/`epsg` are `None`: the coordinates are the model's own engineering frame (roughly
−113…74 in x, −124…164 in y), with no datum, so reprojection and basemap tiles never apply. Unlike a bare
geometry deck, this file does carry one field — `node_z`.

### The `node_z` field — a 3-D surface

`node_z` is a per-node elevation: the deck is a **3-D surface** flattened into a 2-D mesh, with the third
coordinate kept as a data variable. `ds.summary()` returns a pandas table of per-variable statistics — the
cheapest way to characterise a field this large without drawing it.

In [ ]:
ds.summary()

The `node_z` elevations span roughly **1099 → 1666** (model units) over all 752 666 nodes — a real
relief surface, not a flat plate. That single row is the whole mesh summarised; next we need to actually
look at it.

## 6. Why we must crop first

> **A third of a million faces is too much to draw.** Rendering the full 332 200-face wireframe — or a
> full-mesh colour map — would take a long time and produce an enormous, unreadable figure. The right move
> on a mesh this size is to **crop to a small window** and explore the subset. Every pyramids operation
> works identically on the cropped `MeshDataset`, so nothing about the workflow changes — only the scale.

We build a small bounding box around the **median face centroid** (where the mesh is actually dense — its
bounding-box corners are empty) and size it to capture a few thousand faces.

In [ ]:
cx, cy = ds.mesh.face_centroids
mx, my = float(np.median(cx)), float(np.median(cy))
half = 1.5  # model units; tune to land on a few thousand faces
bbox = (mx - half, my - half, mx + half, my + half)
tuple(round(v, 2) for v in bbox)

`crop(bbox=...)` returns a **new** `MeshDataset` containing only the faces inside the box — the
original `ds` is untouched. We print the face count before and after to confirm the window is small enough
to visualise.

In [ ]:
cropped = ds.crop(bbox=bbox)
print(f"faces before / after crop : {ds.n_face}  ->  {cropped.n_face}")
print(f"nodes before / after crop : {ds.n_node}  ->  {cropped.n_node}")

From 332 200 faces down to **~3 360** — a window we can draw instantly. From here on everything runs
on `cropped`, and it behaves exactly like any other `MeshDataset`.

## 7. The wireframe

`plot_outline()` draws the bare cell edges — the quickest confirmation the crop landed on real geometry.
The tile call is guarded by `if cropped.epsg`, so it is skipped for this un-georeferenced mesh.

In [ ]:
_g = cropped.plot_outline()
if cropped.epsg:
    add_tiles(_g.ax, crs=cropped.epsg)
_g.ax.set_title("Abaqus mesh — cropped wireframe")
_g.ax.set_aspect("equal")

An unstructured engineering mesh, finer where the model needs resolution. This small patch is a
faithful slice of the full surface — just small enough to see.

## 8. The third coordinate as a surface

`node_z` is a node-located field, so `plot("node_z", ...)` shades the cropped patch by elevation — a map of
the 3-D relief that the 2-D mesh encodes.

In [ ]:
_g = cropped.plot("node_z", cmap="terrain", colorbar=True, title="Cropped patch — node_z elevation")
if cropped.epsg:
    add_tiles(_g.ax, crs=cropped.epsg)
_g.ax.set_aspect("equal")

The colour gradient is the surface rising and falling across the window — the information that would
be the z-axis in 3-D, carried along as a per-node field.

## 9. A synthesised per-cell field: area

Engineering decks rarely ship a per-cell field we can colour by, so we compute one. `with_variable`
attaches each face's area as a face-located variable, returning a non-destructive copy that `plot` renders
as a filled colour map — the generic pyramids data-map path any per-cell field (depth, velocity, …) would
follow.

In [ ]:
d2 = cropped.with_variable("cell_area", cropped.mesh.face_areas, location="face")
_g = d2.plot("cell_area", cmap="viridis", colorbar=True, title="Cropped patch — cell area")
if cropped.epsg:
    add_tiles(_g.ax, crs=cropped.epsg)
_g.ax.set_aspect("equal")

The map shows the graded element sizes across the patch. `summary()` is the numeric companion to the
colour map:

In [ ]:
d2.summary()

The `cell_area` row reports the min/max/mean element size over the cropped faces, and `node_z` is
carried through the crop too — confirming the refinement gradient the colour map shows.

## 10. GIS operations, for free

Because the cropped mesh is a pyramids object, the pyramids GIS layer applies directly — none of these
methods live in `pyramids-mesh`, which only reads the file. We tour three on the cropped subset.

### Faces as a `GeoDataFrame`

`to_geodataframe()` turns every face into a polygon row of a geopandas `GeoDataFrame` — the standard
vector structure for export to shapefile/GeoPackage or for spatial joins. Plotting it draws every face as
a filled polygon: a vector-native second view of the patch.

In [ ]:
gdf = cropped.to_geodataframe()
ax = gdf.plot(edgecolor="black", linewidth=0.1, facecolor="#cfe8ff")
if cropped.epsg:
    add_tiles(ax, crs=cropped.epsg)
ax.set_aspect("equal"); ax.set_title("Cropped faces as polygons")
gdf.shape

One polygon row per cropped face (~3 360). The CRS is unset — inherited from the mesh's `None` — so
you would assign one before exporting to a shapefile.

### The outer footprint

`bounding_polygon()` is pyramids' one-call shortcut that dissolves the faces into the patch outline. It
assumes **clean topology**, and this is where a real engineering mesh bites: the `.inp` carries a handful
of **degenerate (zero-area) faces** — triangles collapsed when the 3-D surface was flattened — and GEOS
refuses to dissolve them. We catch that, repair the polygons with `shapely.make_valid`, and dissolve the
validated set instead. Both paths produce the same footprint; only the degenerate faces need the detour.

In [ ]:
try:
    footprint = cropped.bounding_polygon()
    how = "bounding_polygon() dissolved cleanly"
except shapely.errors.GEOSException:
    valid = gdf.copy()
    valid["geometry"] = shapely.make_valid(valid.geometry.values)
    footprint = valid.union_all()
    how = "degenerate faces -> repaired with make_valid, then dissolved"

print(how)
ax = gpd.GeoSeries([footprint]).plot(facecolor="none", edgecolor="red", linewidth=1.5)
ax.set_aspect("equal"); ax.set_title("Cropped patch footprint")
pd.Series({"geom_type": footprint.geom_type, "area": round(footprint.area, 2)})

The red outline is the dissolved boundary of the cropped window. The lesson is general: pyramids
offers the clean one-call method, but real FE meshes carry degenerate cells, and the geopandas `make_valid`
+ `union_all` path is the robust fallback when they do.

### Connected components

`connected_components()` labels which faces are reachable from which through shared edges. On a full,
watertight surface this would be a single component — but our window is a **crop**, and a 2-D box slices
the surface into pieces wherever it cuts through, so the subset is naturally fragmented. We colour each
face by its component label to see the islands.

In [ ]:
comp = np.asarray(cropped.connected_components())
d3 = cropped.with_variable("component", comp.astype(float), location="face")
_g = d3.plot("component", cmap="tab20", colorbar=True, title="Cropped patch — connected components")
if cropped.epsg:
    add_tiles(_g.ax, crs=cropped.epsg)
_g.ax.set_aspect("equal")
pd.Series({"n_faces": cropped.n_face, "n_components": int(np.unique(comp).size)})

The crop breaks into dozens of components — the box cut the mesh into separate edge-connected
patches. On the full, uncropped surface these would rejoin; the fragmentation is an artefact of exploring a
slice, and a useful reminder that `crop` gives you a *subset*, not a standalone watertight mesh.

> **Note — xarray interop.** pyramids-core exposes a `to_xarray()` bridge to the xarray ecosystem, but
> these notebooks deliberately stay within pyramids-core's own GDAL-based UGRID I/O and do not import
> `xarray` (nor `h5py`, `netCDF4`, or `scipy`) directly.

## Takeaway

- The `abaqus` driver reads a large keyword `.inp` deck (**752 666 nodes / 332 200 faces**, ~94 MB) into a
  `MeshDataset` in **a few seconds** — scale is not a barrier to reading.
- It is a **3-D surface**: `crs`/`epsg` are `None` (engineering coordinates), and the third coordinate is
  carried as a `node_z` field, with connectivity width `k = 8` for its mixed element types.
- **Drawing the whole mesh is the wrong move.** `crop(bbox=...)` cut it to a few thousand faces, and from
  there the full pyramids toolbox — `plot_outline`, `plot`, `with_variable`, `summary`, `to_geodataframe`,
  `connected_components` — works identically on the subset.
- Real FE meshes carry **degenerate faces**: `bounding_polygon()` can trip GEOS, and the `make_valid` +
  `union_all` dissolve is the robust fallback. Cropping also fragments the surface into components — a
  subset, not a watertight mesh.